# Astro Tabular NN: Broad Grid Trial

This notebook runs a sampled broad grid-search trial around the scout recommendations.

- mandatory CUDA training
- Numba-accelerated margin search
- broad parameter space with random unique candidate sampling


In [ ]:
from pathlib import Path
import sys
import pandas as pd

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "RESEARCH").exists():
    PROJECT_ROOT = PROJECT_ROOT.parents[1]
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

PROJECT_ROOT


In [ ]:
from dataclasses import replace
from RESEARCH.astro_tabular_nn.best_grid_dataset import ensure_best_grid_dataset_path
from RESEARCH.astro_tabular_nn.config import DatasetConfig, ScoutConfig, TrainConfig, with_dataset, with_epochs, with_batch_size
from RESEARCH.astro_tabular_nn.data_utils import load_tabular_dataset
from RESEARCH.astro_tabular_nn.grid_search import GridSearchSpace, run_broad_grid_trial
from RESEARCH.astro_tabular_nn.postrun_report import render_postrun_report


In [ ]:
RUN_TAG = "turning_massive_label_grid"
DATA_START = "2017-11-01"
GRID_EPOCHS = 4
GRID_BATCH_SIZE = 512
GRID_N_TRIALS = 24
GRID_SEEDS = (42,)
GRID_SAMPLE_SEED = 20260208
MODEL_TYPES = ("dcn", "deepfm")

DATASET_PATH = ensure_best_grid_dataset_path(
    run_tag=RUN_TAG,
    data_start=DATA_START,
    use_cache=True,
    verbose=True,
)

space = replace(GridSearchSpace(), model_types=MODEL_TYPES)

base_ds_cfg = DatasetConfig()
ds_cfg = with_dataset(base_ds_cfg, DATASET_PATH)

base_train_cfg = TrainConfig()
train_cfg = with_epochs(base_train_cfg, GRID_EPOCHS)
train_cfg = with_batch_size(train_cfg, GRID_BATCH_SIZE)

scout_cfg = ScoutConfig(train=train_cfg)

ds_cfg, scout_cfg, space


In [ ]:
dataset = load_tabular_dataset(ds_cfg)
results, meta = run_broad_grid_trial(
    dataset=dataset,
    scout_cfg=scout_cfg,
    space=space,
    n_trials=GRID_N_TRIALS,
    seeds=GRID_SEEDS,
    sample_seed=GRID_SAMPLE_SEED,
    verbose=True,
)

postrun_metrics = render_postrun_report(
    results=results,
    run_rank=1,
    title="Grid Trial - Default Post-Run Diagnostics",
)

results.head(20)


In [ ]:
pd.Series(meta['split_summary'])


In [ ]:
display_cols = [
    "run_id", "seed", "model_type", "hidden_dims", "dropout", "cross_layers", "cross_rank", "embed_dim",
    "learning_rate", "weight_decay", "class_weight_power", "label_smoothing", "batch_size",
    "cutoff_kind", "best_epoch", "best_margin", "best_val_score",
    "test_recall_down", "test_recall_up", "test_recall_min", "test_recall_gap", "test_mcc", "test_acc",
    "test_true_up_share", "test_pred_up_share", "test_true_balance_gap_ud", "test_pred_balance_gap_ud",
]
results[display_cols].head(30)


In [ ]:
# Optional save
# out_csv = PROJECT_ROOT / "RESEARCH/reports/astro_tabular_nn_grid_trial_notebook.csv"
# results.to_csv(out_csv, index=False)
# out_csv


## Next Step

Use top-3/5 configs from this table for:

1. longer epochs,
2. multi-seed stability check,
3. walk-forward validation.
